# End-to-End Workflows: Complete User Journeys

## Overview

See how all the pieces fit together in realistic user workflows. This notebook demonstrates complete journeys from data warehouse to graph insights.

**What you'll learn:**
- Complete analyst workflow (mapping → snapshot → instance → query → algorithm)
- Data exploration workflow (schema → counts → samples → stats)
- Chained operations pattern (algorithm results → queries)
- DataFrame-centric analysis workflow

**Prerequisites:**
- Completed: [05_query_graph_algorithms.ipynb](./05_query_graph_algorithms.ipynb)
- Familiarity with mappings, snapshots, instances, and queries

**Time to complete:** 20 minutes

---

**Note:** Centrality and community detection algorithms are thoroughly tested in
`05_query_graph_algorithms.ipynb`. This notebook focuses on demonstrating workflow patterns.

In [ ]:
import os

# Parameters cell - papermill will inject values here
# Note: Uses GRAPH_OLAP_API_URL from environment (set by JupyterHub or local dev)
EXPECTED_NODE_COUNT = 5
EXPECTED_EDGE_COUNT = 6

## Setup

### Test Data Isolation Strategy

1. **Self-Contained**: Creates its own mapping, snapshot, and instance for workflow tests.
2. **Unique Property Names**: Algorithm properties use `wf_*` prefix to avoid conflicts.
3. **Automatic Cleanup**: Uses cleanup framework for reliable resource cleanup.

In [ ]:
import sys
import os

print(f"Python version: {sys.version}")
print(f"GRAPH_OLAP_API_URL: {os.environ.get('GRAPH_OLAP_API_URL', 'not set')}")

In [ ]:
from graph_olap import notebook
from graph_olap.models import EdgeDefinition, NodeDefinition
from graph_olap.models.mapping import PropertyDefinition
from graph_olap.testing import TestPersona
from graph_olap_schemas import WrapperType

# Wake up Starburst Galaxy cluster (auto-suspends after 5 min idle)
notebook.wake_starburst()

# Create test context with automatic cleanup
ctx = notebook.test("WorkflowTest", persona=TestPersona.ANALYST_ALICE)
client = ctx.client

# Define test data
person_node = NodeDefinition(
    label="Person",
    sql="SELECT DISTINCT id, name, age FROM bigquery.graph_olap_e2e.persons",
    primary_key={"name": "id", "type": "STRING"},
    properties=[PropertyDefinition(name="name", type="STRING"), PropertyDefinition(name="age", type="INT64")]
)

knows_edge = EdgeDefinition(
    type="KNOWS",
    from_node="Person",
    to_node="Person",
    sql="SELECT DISTINCT from_id, to_id, since FROM bigquery.graph_olap_e2e.friendships",
    from_key="from_id",
    to_key="to_id",
    properties=[PropertyDefinition(name="since", type="INT64")],
)

print(f"Test run ID: {ctx.run_id}")

## Start Cleanup Context Manager

All resources created from here will be automatically cleaned up.

In [ ]:
# Resources are automatically tracked and cleaned up via ctx
print("Starting Workflow E2E Test - resources will be cleaned up automatically via atexit")

In [ ]:
# Create base mapping using ctx.mapping (auto-tracked)
mapping = ctx.mapping(
    description="Mapping for workflow testing",
    node_definitions=[person_node],
    edge_definitions=[knows_edge],
)
MAPPING_ID = mapping.id
MAPPING_NAME = mapping.name

print(f"Created mapping: {MAPPING_NAME} (id={MAPPING_ID})")

In [ ]:
# SNAPSHOT FUNCTIONALITY DISABLED
# Create instance directly from mapping using create_from_mapping()
# The snapshot is created automatically in the background
print(f"Creating instance directly from mapping (snapshot created automatically)...")

instance = client.instances.create_from_mapping_and_wait(
    mapping_id=MAPPING_ID,
    name=f"WorkflowTest-Instance-{ctx.run_id}",
    wrapper_type=WrapperType.RYUGRAPH,
    timeout=300,  # Longer timeout for snapshot export
    poll_interval=5,
)
INSTANCE_ID = instance.id
INSTANCE_NAME = instance.name
SNAPSHOT_ID = instance.snapshot_id
SNAPSHOT_NAME = f"auto-snapshot-{SNAPSHOT_ID}"
ctx._track('instance', INSTANCE_ID, INSTANCE_NAME)

print(f"Created instance: {INSTANCE_NAME} (id={INSTANCE_ID}, status={instance.status})")
print(f"Auto-created snapshot: {SNAPSHOT_NAME} (id={SNAPSHOT_ID})")

In [ ]:
# SNAPSHOT FUNCTIONALITY DISABLED - instance already created above
# This cell is kept for compatibility but the work is done in the previous cell
print(f"Instance already created: {INSTANCE_NAME} (id={INSTANCE_ID}, status={instance.status})")

In [ ]:
# Connect to instance via ctx
conn = ctx.connect(instance)
print(f"Connected to instance {INSTANCE_ID}")

## 1. Complete Analyst Workflow

Simulates a typical analyst journey: mapping -> snapshot -> instance -> query -> algorithm

In [ ]:
# Step 1: List mappings
mappings = client.mappings.list()
assert len(mappings) > 0, "Should have at least one mapping"
print(f"WF 1.1: Found {len(mappings)} mapping(s)")

# Step 2: Get our mapping
mapping_fetched = client.mappings.get(MAPPING_ID)
assert mapping_fetched.name == MAPPING_NAME
print(f"WF 1.2: Got mapping '{mapping_fetched.name}'")

# Step 3: Get mapping version to understand schema
version = client.mappings.get_version(MAPPING_ID, version=1)
assert len(version.node_definitions) >= 1
assert len(version.edge_definitions) >= 1
print(f"WF 1.3: Mapping has {len(version.node_definitions)} node(s), {len(version.edge_definitions)} edge(s)")

# Step 4: Get our snapshot
snapshot_fetched = client.snapshots.get(SNAPSHOT_ID)
assert snapshot_fetched.status == "ready"
print(f"WF 1.4: Snapshot '{snapshot_fetched.name}' is {snapshot_fetched.status}")

# Step 5: Query the graph
count = conn.query_scalar("MATCH (p:Person) RETURN count(p)")
assert count == EXPECTED_NODE_COUNT
print(f"WF 1.5: Graph has {count} Person nodes")

# Step 6: Run algorithm
execution = conn.algo.pagerank(
    node_label="Person",
    property_name="wf_analyst_pr",
    edge_type="KNOWS"
)
assert execution.status == "completed"
print(f"WF 1.6: PageRank completed, {execution.nodes_updated} nodes updated")

# Track algorithm property for cleanup
ctx._track('graph_properties', conn, {'node_label': 'Person', 'property_names': ['wf_analyst_pr']})

# Step 7: Query algorithm results
df = conn.query_df("""
    MATCH (p:Person)
    RETURN p.name AS name, p.wf_analyst_pr AS pagerank
    ORDER BY p.wf_analyst_pr DESC
""")
assert len(df) == EXPECTED_NODE_COUNT
print("WF 1.7: Retrieved PageRank results")

print("\nWorkflow 1 PASSED: Complete analyst workflow verified")
print(df)

## 2. Data Exploration Workflow

Simulates exploring an unfamiliar graph: schema -> counts -> samples -> stats

In [ ]:
# Step 1: Get schema
schema = conn.get_schema()
assert len(schema.node_labels) >= 1
assert len(schema.relationship_types) >= 1
print(f"WF 2.1: Schema has {len(schema.node_labels)} node label(s), {len(schema.relationship_types)} relationship type(s)")
print(f"  Node labels: {list(schema.node_labels.keys())}")
print(f"  Relationships: {list(schema.relationship_types.keys())}")

# Step 2: Count nodes and edges
node_count = conn.query_scalar("MATCH (n) RETURN count(n)")
edge_count = conn.query_scalar("MATCH ()-[r]->() RETURN count(r)")
assert node_count == EXPECTED_NODE_COUNT
assert edge_count == EXPECTED_EDGE_COUNT
print(f"WF 2.2: Graph size: {node_count} nodes, {edge_count} edges")

# Step 3: Sample data
sample = conn.query("MATCH (p:Person) RETURN p.name, p.age LIMIT 3")
assert sample.row_count == 3
print("WF 2.3: Sample data:")
for row in sample.rows:
    print(f"  {row[0]}: age {row[1]}")

# Step 4: Aggregations with stats
stats = conn.query("""
    MATCH (p:Person)
    RETURN
        min(p.age) AS min_age,
        max(p.age) AS max_age,
        avg(p.age) AS avg_age,
        count(p) AS total
""")
assert stats.row_count == 1
row = stats.rows[0]
assert row[0] == 25, f"Expected min age 25 (Bob), got {row[0]}"
assert row[1] == 35, f"Expected max age 35 (Charlie), got {row[1]}"
assert row[3] == EXPECTED_NODE_COUNT
print(f"WF 2.4: Age stats: min={row[0]}, max={row[1]}, avg={row[2]:.1f}, count={row[3]}")

print("\nWorkflow 2 PASSED: Data exploration workflow verified")

## 3. Chained Operations Pattern

Tests that operations can be chained together fluently

In [ ]:
# Run algorithm and immediately query results
conn.algo.pagerank("Person", "wf_chain_pr", edge_type="KNOWS")

# Track algorithm property for cleanup
ctx._track('graph_properties', conn, {'node_label': 'Person', 'property_names': ['wf_chain_pr']})

top_score = conn.query_scalar("""
    MATCH (p:Person)
    RETURN max(p.wf_chain_pr)
""")
assert top_score > 0

print(f"Workflow 3 PASSED: Chained operations (top score: {top_score:.4f})")

## 4. DataFrame-Centric Workflow

Tests working primarily with DataFrames for data analysis

In [ ]:
# Get data as DataFrame
df = conn.query_df("MATCH (p:Person) RETURN p.name AS name, p.age AS age")

assert len(df) == EXPECTED_NODE_COUNT
assert "name" in df.columns
assert "age" in df.columns

# Basic DataFrame operations
avg_age = df["age"].mean()
max_age = df["age"].max()
min_age = df["age"].min()

print("Workflow 4 PASSED: DataFrame-centric workflow")
print(f"  Shape: {df.shape}")
print(f"  Age stats: min={min_age}, max={max_age}, avg={avg_age:.1f}")
print(df)

## Summary

In [ ]:
# Cleanup is handled automatically by ctx via atexit
# For interactive use, you can call ctx.cleanup() manually

print("\nCleanup will happen automatically via atexit")

In [ ]:
# Workflow tests completed
print("Workflow tests completed")

In [ ]:
print("\n" + "="*60)
print("WORKFLOW E2E TESTS COMPLETED!")
print("="*60)
print("\nValidated:")
print("  1. Complete Analyst Workflow:")
print("    - Mapping -> Snapshot -> Instance -> Query -> Algorithm")
print("  2. Data Exploration Workflow:")
print("    - Schema -> Counts -> Samples -> Stats")
print("  3. Chained Operations Pattern:")
print("    - Algorithm execution followed by query")
print("  4. DataFrame-Centric Workflow:")
print("    - Query to DataFrame -> Analysis")
print("\nNote: Centrality and community detection algorithms are")
print("      tested in sdk_algorithm_test.ipynb")
print("\nAll resources will be cleaned up automatically via atexit")